# 06 Silver Observation Clean

## Purpose

This notebook creates the Silver Observation table from the Bronze raw Observation FHIR table.

## What We Are Doing

We will:
1. Read `healthcare_catalog.bronze.observation_raw`
2. Extract lab and vital sign fields
3. Flatten observation data
4. Standardize values and timestamps
5. Save it as `healthcare_catalog.silver.observation_clean`

## Why We Are Doing This

Observation resources contain:
- vitals
- laboratory results
- measurements
- patient monitoring data

This table is one of the most important healthcare analytics datasets.

It will later support:
- risk prediction
- diabetes analytics
- cardiovascular analytics
- sepsis monitoring
- patient deterioration models

## Expected Final Output

A clean Silver table:

`healthcare_catalog.silver.observation_clean`

Expected columns:
- observation_id
- patient_id
- encounter_id
- observation_description
- observation_category
- observation_value
- observation_unit
- observation_datetime

## Step 1 — Import PySpark Functions

### What We Are Doing

We are importing PySpark SQL functions.

### Why We Are Doing This

We need Spark functions to extract nested FHIR Observation fields.

### Expected Output

Spark functions available for this notebook.

In [0]:
from pyspark.sql.functions import *

## Step 2 — Read Bronze Observation Table

### What We Are Doing

We are reading the Bronze Observation table.

### Why We Are Doing This

The Bronze layer contains raw nested FHIR Observation resources.

### Expected Output

A DataFrame named:

`observation_raw_df`

In [0]:
observation_raw_df = spark.table(
    "healthcare_catalog.bronze.observation_raw"
)

print("Bronze observation_raw table loaded successfully.")

Bronze observation_raw table loaded successfully.


## Step 3 — Inspect Observation Raw Schema

### What We Are Doing

We are printing the Observation schema.

### Why We Are Doing This

FHIR Observation resources are deeply nested and highly variable.

We need to identify:
- patient references
- encounter references
- observation values
- units
- categories
- timestamps

### Expected Output

You should see fields such as:
- resource.id
- resource.subject.reference
- resource.encounter.reference
- resource.code
- resource.category
- resource.valueQuantity
- resource.effectiveDateTime

In [0]:
observation_raw_df.printSchema()

root
 |-- fullUrl: string (nullable = true)
 |-- resourceType: string (nullable = true)
 |-- resource: struct (nullable = true)
 |    |-- abatementDateTime: string (nullable = true)
 |    |-- active: boolean (nullable = true)
 |    |-- activity: array (nullable = true)
 |    |    |-- element: struct (containsNull = true)
 |    |    |    |-- detail: struct (nullable = true)
 |    |    |    |    |-- code: struct (nullable = true)
 |    |    |    |    |    |-- coding: array (nullable = true)
 |    |    |    |    |    |    |-- element: struct (containsNull = true)
 |    |    |    |    |    |    |    |-- code: string (nullable = true)
 |    |    |    |    |    |    |    |-- display: string (nullable = true)
 |    |    |    |    |    |    |    |-- system: string (nullable = true)
 |    |    |    |    |    |-- text: string (nullable = true)
 |    |    |    |    |-- location: struct (nullable = true)
 |    |    |    |    |    |-- display: string (nullable = true)
 |    |    |    |    |-- statu

## Step 4 — Extract Clean Observation Columns

### What We Are Doing

We are extracting vitals and laboratory fields from nested FHIR Observation resources.

### Why We Are Doing This

FHIR Observation resources contain:
- blood pressure
- BMI
- glucose
- oxygen saturation
- cholesterol
- laboratory results
- patient monitoring values

Analytics and ML models require flattened observation tables.

### Fields We Will Extract

- observation_id
- patient_reference
- encounter_reference
- observation_description
- observation_category
- observation_value
- observation_unit
- observation_datetime

### Expected Output

A flattened DataFrame named:

`observation_clean_df`

In [0]:
observation_clean_df = observation_raw_df.select(

    col("resource.id").alias("observation_id"),

    col("resource.subject.reference").alias("patient_reference"),

    col("resource.encounter.reference").alias("encounter_reference"),

    get_json_object(col("resource.code"), "$.text").alias("observation_description"),

    get_json_object(col("resource.category")[0], "$.coding[0].code").alias("observation_category"),

    col("resource.valueQuantity.value").cast("double").alias("observation_value"),

    col("resource.valueQuantity.unit").alias("observation_unit"),

    col("resource.effectiveDateTime").alias("observation_datetime")
)

print("Observation clean DataFrame created successfully.")

Observation clean DataFrame created successfully.


## Step 5 — Convert Observation Timestamp

### What We Are Doing

We are converting observation timestamps into Spark timestamp format.

### Why We Are Doing This

Time-series healthcare analytics require timestamp data types.

This supports:
- patient monitoring
- longitudinal analysis
- deterioration tracking
- temporal ML features

### Expected Output

Observation timestamps converted successfully.

In [0]:
observation_clean_df = observation_clean_df.withColumn(
    "observation_datetime",
    to_timestamp(col("observation_datetime"))
)

print("Observation timestamps converted successfully.")

Observation timestamps converted successfully.


## Step 6 — Extract Clean Patient and Encounter IDs

### What We Are Doing

We are extracting clean UUIDs from FHIR reference fields.

### Why We Are Doing This

FHIR references contain:
- urn:uuid:
- ResourceType/ID

We need normalized IDs for joins across tables.

### Expected Output

New columns:
- patient_id
- encounter_id

In [0]:
observation_clean_df = observation_clean_df.withColumn(
    "patient_id",

    regexp_extract(
        col("patient_reference"),
        r'urn:uuid:(.*)',
        1
    )
)

observation_clean_df = observation_clean_df.withColumn(
    "encounter_id",

    regexp_extract(
        col("encounter_reference"),
        r'urn:uuid:(.*)',
        1
    )
)

print("Patient and encounter IDs extracted successfully.")

Patient and encounter IDs extracted successfully.


## Step 7 — Inspect Clean Observation Data

### What We Are Doing

We are displaying the flattened observation table.

### Why We Are Doing This

We need to verify:
- vitals/labs extracted correctly
- values appear correctly
- units are preserved
- timestamps converted properly

### Expected Output

A clean observation-level healthcare table.

In [0]:
display(observation_clean_df)

observation_id,patient_reference,encounter_reference,observation_description,observation_category,observation_value,observation_unit,observation_datetime,patient_id,encounter_id
75b4cb64-baea-c627-4d12-282f376d9d32,urn:uuid:29b21635-3f05-3634-1941-4122fb2471ce,urn:uuid:91f70453-9d56-e8d2-3f2b-60865bf0dc1f,Body Height,vital-signs,182.6,cm,2011-12-20T00:25:24.000Z,29b21635-3f05-3634-1941-4122fb2471ce,91f70453-9d56-e8d2-3f2b-60865bf0dc1f
0d756fad-8345-212c-d079-55fead3620e8,urn:uuid:29b21635-3f05-3634-1941-4122fb2471ce,urn:uuid:91f70453-9d56-e8d2-3f2b-60865bf0dc1f,Pain severity - 0-10 verbal numeric rating [Score] - Reported,vital-signs,3.0,{score},2011-12-20T00:25:24.000Z,29b21635-3f05-3634-1941-4122fb2471ce,91f70453-9d56-e8d2-3f2b-60865bf0dc1f
d80697a0-2007-9e12-4afa-9439ba4d67b0,urn:uuid:29b21635-3f05-3634-1941-4122fb2471ce,urn:uuid:91f70453-9d56-e8d2-3f2b-60865bf0dc1f,Body Weight,vital-signs,84.7,kg,2011-12-20T00:25:24.000Z,29b21635-3f05-3634-1941-4122fb2471ce,91f70453-9d56-e8d2-3f2b-60865bf0dc1f
1354da6e-f866-9e62-0507-6d6956484f7d,urn:uuid:29b21635-3f05-3634-1941-4122fb2471ce,urn:uuid:91f70453-9d56-e8d2-3f2b-60865bf0dc1f,Body Mass Index,vital-signs,25.4,kg/m2,2011-12-20T00:25:24.000Z,29b21635-3f05-3634-1941-4122fb2471ce,91f70453-9d56-e8d2-3f2b-60865bf0dc1f
58a6ee37-5542-3c75-dcb5-3641787aa62f,urn:uuid:29b21635-3f05-3634-1941-4122fb2471ce,urn:uuid:91f70453-9d56-e8d2-3f2b-60865bf0dc1f,Blood Pressure,vital-signs,null,null,2011-12-20T00:25:24.000Z,29b21635-3f05-3634-1941-4122fb2471ce,91f70453-9d56-e8d2-3f2b-60865bf0dc1f
408f2553-8702-2039-022e-bd16cef115f4,urn:uuid:29b21635-3f05-3634-1941-4122fb2471ce,urn:uuid:91f70453-9d56-e8d2-3f2b-60865bf0dc1f,Heart rate,vital-signs,85.0,/min,2011-12-20T00:25:24.000Z,29b21635-3f05-3634-1941-4122fb2471ce,91f70453-9d56-e8d2-3f2b-60865bf0dc1f
79e6ecef-aad3-5c34-bf4c-fc92c0f2f3eb,urn:uuid:29b21635-3f05-3634-1941-4122fb2471ce,urn:uuid:91f70453-9d56-e8d2-3f2b-60865bf0dc1f,Respiratory rate,vital-signs,15.0,/min,2011-12-20T00:25:24.000Z,29b21635-3f05-3634-1941-4122fb2471ce,91f70453-9d56-e8d2-3f2b-60865bf0dc1f
31eb20db-9178-96c9-c5fb-c5a3857fd509,urn:uuid:29b21635-3f05-3634-1941-4122fb2471ce,urn:uuid:91f70453-9d56-e8d2-3f2b-60865bf0dc1f,Tobacco smoking status NHIS,survey,null,null,2011-12-20T00:25:24.000Z,29b21635-3f05-3634-1941-4122fb2471ce,91f70453-9d56-e8d2-3f2b-60865bf0dc1f
75a948a3-061d-03ba-61ea-94b2cf170d40,urn:uuid:29b21635-3f05-3634-1941-4122fb2471ce,urn:uuid:91f70453-9d56-e8d2-3f2b-60865bf0dc1f,"Protocol for Responding to and Assessing Patients' Assets, Risks, and Experiences [PRAPARE]",survey,null,null,2011-12-20T01:13:05.000Z,29b21635-3f05-3634-1941-4122fb2471ce,91f70453-9d56-e8d2-3f2b-60865bf0dc1f
ff5b11dc-0b9e-f1ee-3559-0619d2557cdd,urn:uuid:29b21635-3f05-3634-1941-4122fb2471ce,urn:uuid:91f70453-9d56-e8d2-3f2b-60865bf0dc1f,Patient Health Questionnaire 2 item (PHQ-2) total score [Reported],survey,0.0,{score},2011-12-20T01:51:09.000Z,29b21635-3f05-3634-1941-4122fb2471ce,91f70453-9d56-e8d2-3f2b-60865bf0dc1f


## Step 8 — Check Most Common Observations

### What We Are Doing

We are counting the most common observation types.

### Why We Are Doing This

This helps us understand:
- what vitals/labs exist
- data quality
- future ML opportunities

### Expected Output

Top observation frequency table.

In [0]:
display(

    observation_clean_df.groupBy(
        "observation_description"
    ).count().orderBy(
        desc("count")
    )

)

observation_description,count
Pain severity - 0-10 verbal numeric rating [Score] - Reported,7146
Blood Pressure,6103
Body Weight,5956
Heart rate,5870
Respiratory rate,5870
Body Height,5732
Tobacco smoking status NHIS,5700
Body Mass Index,5187
"Protocol for Responding to and Assessing Patients' Assets, Risks, and Experiences [PRAPARE]",3835
Patient Health Questionnaire 2 item (PHQ-2) total score [Reported],2985


## Step 9 — Check Null Values

### What We Are Doing

We are validating missing observation fields.

### Why We Are Doing This

Clinical observations frequently contain:
- missing values
- missing units
- missing timestamps

Silver validation is critical before Gold analytics.

### Expected Output

Null count summary.

In [0]:
display(

    observation_clean_df.select(

        [
            sum(col(column_name).isNull().cast("int")).alias(column_name)

            for column_name in observation_clean_df.columns
        ]

    )

)

observation_id,patient_reference,encounter_reference,observation_description,observation_category,observation_value,observation_unit,observation_datetime,patient_id,encounter_id
0,0,0,0,0,20392,20392,0,0,0


## Step 10 — Save Silver Observation Table

### What We Are Doing

We are saving the clean observation table into the Silver layer.

### Why We Are Doing This

The Silver layer stores:
- cleaned
- normalized
- analytics-ready

vitals and laboratory data.

This table will later support:
- patient risk scoring
- clinical AI
- deterioration prediction
- diabetes analytics
- cardiovascular analytics

### Expected Output

A Delta table:

`healthcare_catalog.silver.observation_clean`

In [0]:
observation_clean_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("healthcare_catalog.silver.observation_clean")

print("Silver observation_clean table saved successfully.")

Silver observation_clean table saved successfully.


In [0]:
spark.sql("""
SHOW TABLES IN healthcare_catalog.silver
""").show(truncate=False)

+--------+-----------------+-----------+
|database|tableName        |isTemporary|
+--------+-----------------+-----------+
|silver  |condition_clean  |false      |
|silver  |encounter_clean  |false      |
|silver  |observation_clean|false      |
|silver  |patient_clean    |false      |
+--------+-----------------+-----------+

